BEM参考解

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from numba import njit, prange
import time

# ==========================================
# 1. 物理参数与网格参数
# ==========================================
R_major = 0.5        
R_minor = 0.25       
L = 2.0              

tn = 1.0             
E = 1000.0           
nu = 0.3             

nu_eff = nu / (1.0 + nu)  
G_shear = E / (2.0 * (1.0 + nu))

print(f"=== 多核 BEM 求解有限域椭圆开孔受拉板问题 ===")

# ==========================================
# 2. 生成多连通域边界网格 (外方内椭圆)
# ==========================================
N_hole = 240          
N_side = 61          
N = N_side * 4 + N_hole

nodes = np.zeros((N, 2))       # 源点 (单元中点)
normals = np.zeros((N, 2))
lengths = np.zeros(N)
bnd_tag = np.zeros(N, dtype=int) 
endpoints = np.zeros((N, 4))   # 记录几何端点用于画图

idx = 0

def add_segment(x1, y1, x2, y2, tag):
    global idx
    nodes[idx] = [(x1 + x2) / 2, (y1 + y2) / 2] # 源点在中间
    endpoints[idx] = [x1, y1, x2, y2]           # 记录几何端点
    vx, vy = x2 - x1, y2 - y1
    L_j = np.hypot(vx, vy)
    lengths[idx] = L_j
    normals[idx] = [vy / L_j, -vx / L_j] 
    bnd_tag[idx] = tag
    idx += 1

# 外边界
coords_c = np.linspace(-L/2, L/2, N_side + 1)
for i in range(N_side): add_segment(coords_c[i], -L/2, coords_c[i+1], -L/2, 1)        
for i in range(N_side): add_segment(L/2, coords_c[i], L/2, coords_c[i+1], 2)          
for i in range(N_side): add_segment(coords_c[N_side-i], L/2, coords_c[N_side-i-1], L/2, 3) 
for i in range(N_side): add_segment(-L/2, coords_c[N_side-i], -L/2, coords_c[N_side-i-1], 4) 

# 内椭圆孔
theta = np.linspace(2 * np.pi, 0, N_hole + 1)
for i in range(N_hole):
    x1, y1 = R_major * np.cos(theta[i]), R_minor * np.sin(theta[i])
    x2, y2 = R_major * np.cos(theta[i+1]), R_minor * np.sin(theta[i+1])
    add_segment(x1, y1, x2, y2, 0)

# ==========================================
# 🌟 可视化：源点与几何端点的分布
# ==========================================
plt.figure(figsize=(7, 7))
# 画边界的蓝色几何端点 (Integration boundaries)
for i in range(N):
    x1, y1, x2, y2 = endpoints[i]
    plt.plot([x1, x2], [y1, y2], 'k-', lw=0.5, alpha=0.5)
    plt.plot(x1, y1, 'bo', markersize=3)
    plt.plot(x2, y2, 'bo', markersize=3)

# 画红色的源点 (Source points)
plt.plot(nodes[:, 0], nodes[:, 1], 'ro', markersize=3, label='Source points (Midpoints)')
plt.plot([], [], 'bo', markersize=3, label='Geometric nodes (Integration bounds)')

plt.legend(loc='upper right')
plt.title("BEM Mesh: Source Points vs Geometric Nodes")
plt.axis('equal')
plt.show()

# ==========================================
# 3. 组装与求解 (严格保持你原来的逻辑)
# ==========================================
C1 = 1.0 / (8.0 * np.pi * G_shear * (1.0 - nu_eff))
C2 = 3.0 - 4.0 * nu_eff
C3 = -1.0 / (4.0 * np.pi * (1.0 - nu_eff))
C4 = 1.0 - 2.0 * nu_eff
gauss_xi, gauss_w = np.polynomial.legendre.leggauss(8)

@njit(parallel=True)
def assemble_HG_numba(nodes, normals, lengths, gauss_xi, gauss_w, C1, C2, C3, C4):
    N = nodes.shape[0]
    H, G = np.zeros((2*N, 2*N)), np.zeros((2*N, 2*N))
    n_gauss = len(gauss_xi)
    
    for i in prange(N):
        x_source = nodes[i]
        for j in range(N):
            L_j = lengths[j]
            nx, ny = normals[j, 0], normals[j, 1]
            tx, ty = -ny, nx
            r0, r1, c0, c1 = 2*i, 2*i+1, 2*j, 2*j+1
            
            if i == j:
                H[r0, c0] = 0.5; H[r0, c1] = 0.0
                H[r1, c0] = 0.0; H[r1, c1] = 0.5
                g00 = C1 * L_j * (C2 * (1.0 - np.log(L_j/2.0)) + 1.0)
                g11 = C1 * L_j * (C2 * (1.0 - np.log(L_j/2.0)))
                G[r0, c0] = tx*(g00*tx) + nx*(g11*nx); G[r0, c1] = tx*(g00*ty) + nx*(g11*ny)
                G[r1, c0] = ty*(g00*tx) + ny*(g11*nx); G[r1, c1] = ty*(g00*ty) + ny*(g11*ny)
            else:
                y_center = nodes[j]
                H00 = H01 = H10 = H11 = G00 = G01 = G10 = G11 = 0.0
                for g in range(n_gauss):
                    rx = y_center[0] + 0.5 * L_j * gauss_xi[g] * tx - x_source[0]
                    ry = y_center[1] + 0.5 * L_j * gauss_xi[g] * ty - x_source[1]
                    r = np.sqrt(rx*rx + ry*ry)
                    rdx, rdy = rx/r, ry/r
                    dr_dn = rdx*nx + rdy*ny
                    log_1_r = np.log(1.0/r)
                    hw = 0.5 * L_j * gauss_w[g]
                    
                    G00 += C1 * (C2 * log_1_r + rdx*rdx) * hw
                    G01 += C1 * (rdx*rdy) * hw
                    G10 += C1 * (rdx*rdy) * hw
                    G11 += C1 * (C2 * log_1_r + rdy*rdy) * hw
                    
                    H00 += C3/r * (dr_dn*(C4 + 2*rdx*rdx) - C4*(rdx*nx - rdx*nx)) * hw
                    H01 += C3/r * (dr_dn*(2*rdx*rdy) - C4*(rdx*ny - rdy*nx)) * hw
                    H10 += C3/r * (dr_dn*(2*rdy*rdx) - C4*(rdy*nx - rdx*ny)) * hw
                    H11 += C3/r * (dr_dn*(C4 + 2*rdy*rdy) - C4*(rdy*ny - rdy*ny)) * hw
                    
                H[r0, c0] = H00; H[r0, c1] = H01; H[r1, c0] = H10; H[r1, c1] = H11
                G[r0, c0] = G00; G[r0, c1] = G01; G[r1, c0] = G10; G[r1, c1] = G11
    return H, G

t0 = time.time()
print("组装系统矩阵...")
H, G = assemble_HG_numba(nodes, normals, lengths, gauss_xi, gauss_w, C1, C2, C3, C4)

A, B = np.zeros((2*N, 2*N)), np.zeros(2*N)
u_known, u_val = np.zeros(2*N, dtype=bool), np.zeros(2*N)
t_known, t_val = np.zeros(2*N, dtype=bool), np.zeros(2*N)

for i in range(N):
    dof_x, dof_y = 2*i, 2*i+1
    tag = bnd_tag[i]
    if tag == 0:     
        t_known[dof_x] = True; t_val[dof_x] = 0.0
        t_known[dof_y] = True; t_val[dof_y] = 0.0
    elif tag == 1:   
        u_known[dof_y] = True; u_val[dof_y] = 0.0  
        t_known[dof_x] = True; t_val[dof_x] = 0.0  
        if abs(nodes[i, 0]) < 1e-5:
            t_known[dof_x] = False
            u_known[dof_x] = True; u_val[dof_x] = 0.0
    elif tag == 2:   
        t_known[dof_x] = True; t_val[dof_x] = 0.0
        t_known[dof_y] = True; t_val[dof_y] = 0.0
    elif tag == 3:   
        t_known[dof_x] = True; t_val[dof_x] = 0.0
        t_known[dof_y] = True; t_val[dof_y] = tn
    elif tag == 4:   
        t_known[dof_x] = True; t_val[dof_x] = 0.0
        t_known[dof_y] = True; t_val[dof_y] = 0.0

for k in range(2*N):
    if u_known[k]:
        A[:, k] = -G[:, k]; B -= H[:, k] * u_val[k]
    else:
        A[:, k] = H[:, k]; B += G[:, k] * t_val[k]

print("求解线性方程组...")
X = np.linalg.solve(A, B)
u_bnd = np.where(u_known, u_val, X)
t_bnd = np.where(u_known, X, t_val)

# ==========================================
# 4. 提取观测区域内场数据：BEM 位移 + 局部最小二乘应力恢复
# ==========================================
@njit(parallel=True)
def calc_internal_u_numba(pts, nodes, normals, lengths, gauss_xi, gauss_w, C1, C2, C3, C4, u_bnd, t_bnd):
    """由 Somigliana 位移恒等式计算域内位移。"""
    N_pts, N = pts.shape[0], nodes.shape[0]
    n_gauss = len(gauss_xi)
    u_out = np.zeros((N_pts, 2))

    for q in prange(N_pts):
        xi_x, xi_y = pts[q, 0], pts[q, 1]
        u_x = 0.0
        u_y = 0.0
        for j in range(N):
            y_center = nodes[j]
            L_j = lengths[j]
            nx, ny = normals[j, 0], normals[j, 1]
            tx, ty = -ny, nx
            U00 = U01 = U10 = U11 = 0.0
            T00 = T01 = T10 = T11 = 0.0

            for g in range(n_gauss):
                rx = y_center[0] + 0.5 * L_j * gauss_xi[g] * tx - xi_x
                ry = y_center[1] + 0.5 * L_j * gauss_xi[g] * ty - xi_y
                r = np.sqrt(rx*rx + ry*ry)
                if r < 1.0e-14:
                    continue
                rdx, rdy = rx/r, ry/r
                dr_dn = rdx*nx + rdy*ny
                log_1_r = np.log(1.0/r)
                hw = 0.5 * L_j * gauss_w[g]

                U00 += C1 * (C2 * log_1_r + rdx*rdx) * hw
                U01 += C1 * (rdx*rdy) * hw
                U10 += C1 * (rdx*rdy) * hw
                U11 += C1 * (C2 * log_1_r + rdy*rdy) * hw

                T00 += C3/r * (dr_dn*(C4 + 2*rdx*rdx) - C4*(rdx*nx - rdx*nx)) * hw
                T01 += C3/r * (dr_dn*(2*rdx*rdy) - C4*(rdx*ny - rdy*nx)) * hw
                T10 += C3/r * (dr_dn*(2*rdy*rdx) - C4*(rdy*nx - rdx*ny)) * hw
                T11 += C3/r * (dr_dn*(C4 + 2*rdy*rdy) - C4*(rdy*ny - rdy*ny)) * hw

            uj_x, uj_y = u_bnd[2*j], u_bnd[2*j+1]
            tj_x, tj_y = t_bnd[2*j], t_bnd[2*j+1]
            u_x += (U00*tj_x + U01*tj_y) - (T00*uj_x + T01*uj_y)
            u_y += (U10*tj_x + U11*tj_y) - (T10*uj_x + T11*uj_y)

        u_out[q, 0], u_out[q, 1] = u_x, u_y
    return u_out


@njit(parallel=True)
def recover_gradients_lstsq_numba(Ux, Uy, plot_mask, fit_mask, h, radius, sigma):
    """局部加权线性最小二乘恢复位移梯度。

    直接有限差分会在椭圆孔边和外边界附近放大 BEM 常单元误差；
    这里在每个点附近解 2x2 法方程，得到平滑且局部守形的 du/dx, du/dy。
    """
    ny, nx = Ux.shape
    grad = np.full((ny, nx, 4), np.nan)  # [dux_dx, dux_dy, duy_dx, duy_dy]
    sigma2 = sigma * sigma

    for j in prange(ny):
        for i in range(nx):
            if not plot_mask[j, i]:
                continue
            if not (np.isfinite(Ux[j, i]) and np.isfinite(Uy[j, i])):
                continue

            sxx = 0.0
            sxy = 0.0
            syy = 0.0
            bx_u = 0.0
            by_u = 0.0
            bx_v = 0.0
            by_v = 0.0
            count = 0
            u0 = Ux[j, i]
            v0 = Uy[j, i]

            j0 = max(0, j - radius)
            j1 = min(ny, j + radius + 1)
            i0 = max(0, i - radius)
            i1 = min(nx, i + radius + 1)

            for jj in range(j0, j1):
                for ii in range(i0, i1):
                    if jj == j and ii == i:
                        continue
                    if not fit_mask[jj, ii]:
                        continue
                    if not (np.isfinite(Ux[jj, ii]) and np.isfinite(Uy[jj, ii])):
                        continue

                    dx = (ii - i) * h
                    dy = (jj - j) * h
                    rr = (dx*dx + dy*dy) / (h*h)
                    w = np.exp(-0.5 * rr / sigma2)
                    du = Ux[jj, ii] - u0
                    dv = Uy[jj, ii] - v0

                    sxx += w * dx * dx
                    sxy += w * dx * dy
                    syy += w * dy * dy
                    bx_u += w * dx * du
                    by_u += w * dy * du
                    bx_v += w * dx * dv
                    by_v += w * dy * dv
                    count += 1

            det = sxx * syy - sxy * sxy
            if count >= 6 and abs(det) > 1.0e-28:
                grad[j, i, 0] = (bx_u * syy - by_u * sxy) / det
                grad[j, i, 1] = (sxx * by_u - sxy * bx_u) / det
                grad[j, i, 2] = (bx_v * syy - by_v * sxy) / det
                grad[j, i, 3] = (sxx * by_v - sxy * bx_v) / det

    return grad


print("提取近场 BEM 位移，并进行局部最小二乘应力恢复...")
L_plot = L / 2.0
grid_size = 220
BEM_PLOT_MASK_LEVEL = 1.03
BEM_FIT_MASK_LEVEL = 1.03

# 不在外边界精确取点，避免内部表示式在边界处退化；extent 仍为 [-1,1]，图像不会截断。
coords_1d = np.linspace(-L_plot, L_plot, grid_size + 2)[1:-1]
grid_h = coords_1d[1] - coords_1d[0]
XX, YY = np.meshgrid(coords_1d, coords_1d)
internal_pts_raw = np.c_[XX.ravel(), YY.ravel()]

ellipse_val_grid = (XX / R_major)**2 + (YY / R_minor)**2
eval_mask_grid = ellipse_val_grid >= 1.0001
plot_mask_grid = ellipse_val_grid >= BEM_PLOT_MASK_LEVEL
fit_mask_grid = ellipse_val_grid >= BEM_FIT_MASK_LEVEL

t2 = time.time()
eval_pts = internal_pts_raw[eval_mask_grid.reshape(-1)]
u_eval = calc_internal_u_numba(
    eval_pts, nodes, normals, lengths,
    gauss_xi, gauss_w, C1, C2, C3, C4,
    u_bnd, t_bnd,
)
print(f"BEM 内点位移提取耗时: {time.time() - t2:.2f} s")

U_X_grid = np.full((grid_size, grid_size), np.nan)
U_Y_grid = np.full((grid_size, grid_size), np.nan)
U_X_grid[eval_mask_grid] = u_eval[:, 0]
U_Y_grid[eval_mask_grid] = u_eval[:, 1]

t3 = time.time()
grad_grid = recover_gradients_lstsq_numba(
    U_X_grid, U_Y_grid,
    plot_mask_grid, fit_mask_grid,
    grid_h, 5, 2.5,
)
print(f"BEM 应力恢复耗时: {time.time() - t3:.2f} s")

eps_xx_grid = grad_grid[:, :, 0]
eps_yy_grid = grad_grid[:, :, 3]
eps_xy_grid = 0.5 * (grad_grid[:, :, 1] + grad_grid[:, :, 2])

mu_lame = E / (2.0 * (1.0 + nu))
E_ps = E / (1.0 - nu**2)
sig_xx_grid = E_ps * (eps_xx_grid + nu * eps_yy_grid)
sig_yy_grid = E_ps * (eps_yy_grid + nu * eps_xx_grid)
sig_xy_grid = 2.0 * mu_lame * eps_xy_grid
sig_zz_grid = 0.0

VM_grid = np.sqrt(
    0.5 * (
        (sig_xx_grid - sig_yy_grid)**2
        + (sig_yy_grid - sig_zz_grid)**2
        + (sig_zz_grid - sig_xx_grid)**2
    )
    + 3.0 * sig_xy_grid**2
)

U_X_grid[~plot_mask_grid] = np.nan
U_Y_grid[~plot_mask_grid] = np.nan
sig_xx_grid[~plot_mask_grid] = np.nan
sig_yy_grid[~plot_mask_grid] = np.nan
sig_xy_grid[~plot_mask_grid] = np.nan
VM_grid[~plot_mask_grid] = np.nan

# 后续 KM-PIKFNN 误差分析使用这些变量。确保 internal_pts、u_int、von_mises 一一对应。
valid_mask_grid = plot_mask_grid
valid_mask = valid_mask_grid.reshape(-1)
internal_pts = internal_pts_raw[valid_mask]
u_int = np.column_stack([U_X_grid[valid_mask_grid], U_Y_grid[valid_mask_grid]])
sig_xx = sig_xx_grid[valid_mask_grid]
sig_yy = sig_yy_grid[valid_mask_grid]
sig_xy = sig_xy_grid[valid_mask_grid]
von_mises = VM_grid[valid_mask_grid]

U_X_map = U_X_grid.reshape(-1)
U_Y_map = U_Y_grid.reshape(-1)
VM_map = VM_grid.reshape(-1)

BEM_VM_VMIN = 0.3
BEM_VM_VMAX = float(np.nanmax(von_mises))
print(
    f"BEM von Mises range: min={np.nanmin(von_mises):.4e} Pa, "
    f"max={np.nanmax(von_mises):.4e} Pa"
)

# ==========================================
# 5. 绘图出图：保留孔左右端与底部角点集中，不画截断虚线
# ==========================================
import matplotlib.gridspec as gridspec

extent = [-L_plot, L_plot, -L_plot, L_plot]
hole_cover_scale = np.sqrt(BEM_PLOT_MASK_LEVEL)


def _cover_hole(ax):
    ax.add_patch(
        Ellipse(
            (0, 0),
            2 * R_major * hole_cover_scale,
            2 * R_minor * hole_cover_scale,
            facecolor="white",
            edgecolor="none",
            zorder=5,
        )
    )


fig = plt.figure(figsize=(16, 4.5), dpi=150)
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

ax0 = fig.add_subplot(gs[0, 0])
im0 = ax0.imshow(U_X_grid * 1e3, origin='lower', extent=extent, cmap='jet',
                 vmin=-0.8, vmax=0.8, interpolation='bilinear')
ax0.set_xlabel('X (m)'); ax0.set_ylabel('Y (m)')
fig.colorbar(im0, ax=ax0, fraction=0.046, pad=0.04)
ax0.set_title('BEM: $U_x$ (mm)')
_cover_hole(ax0)

ax1 = fig.add_subplot(gs[0, 1])
im1 = ax1.imshow(U_Y_grid * 1e3, origin='lower', extent=extent, cmap='jet',
                 vmin=0.0, vmax=3.5, interpolation='bilinear')
ax1.set_xlabel('X (m)'); ax1.set_ylabel('Y (m)')
fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
ax1.set_title('BEM: $U_y$ (mm)')
_cover_hole(ax1)

ax2 = fig.add_subplot(gs[0, 2])
im2 = ax2.imshow(VM_grid, origin='lower', extent=extent, cmap='jet',
                 vmin=BEM_VM_VMIN, vmax=BEM_VM_VMAX, interpolation='bilinear')
ax2.set_xlabel('X (m)'); ax2.set_ylabel('Y (m)')
cbar2 = fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
cbar2.set_label('Pa')
ax2.set_title('BEM: $\sigma_{von}$ (Pa)')
_cover_hole(ax2)

for ax in [ax0, ax1, ax2]:
    ax.set_xlim(-L_plot, L_plot)
    ax.set_ylim(-L_plot, L_plot)
    ax.set_aspect('equal', adjustable='box')
    ax.tick_params(axis='both', which='major', direction='in', top=True, right=True)
    for spine in ax.spines.values():
        spine.set_visible(False)

plt.show()

# 论文中可直接使用的单张 von Mises 云图
fig_vm, ax_vm = plt.subplots(figsize=(4.6, 3.6), dpi=180)
im_vm = ax_vm.imshow(VM_grid, origin='lower', extent=extent, cmap='jet',
                     vmin=BEM_VM_VMIN, vmax=BEM_VM_VMAX, interpolation='bilinear')
ax_vm.set_xlim(-L_plot, L_plot)
ax_vm.set_ylim(-L_plot, L_plot)
ax_vm.set_aspect('equal', adjustable='box')
ax_vm.set_title(r'$\sigma_{von}\ (\mathrm{Pa})$', fontsize=13, pad=6)
ax_vm.tick_params(axis='both', which='major', direction='in', top=True, right=True, labelsize=9)
_cover_hole(ax_vm)
cbar_vm = fig_vm.colorbar(im_vm, ax=ax_vm, fraction=0.046, pad=0.04)
cbar_vm.ax.tick_params(labelsize=9)
for spine in ax_vm.spines.values():
    spine.set_visible(False)
plt.tight_layout()
plt.show()

KM-PIKFNN 点力基本解法

该单元将原来的复值 MLP/极点展开替换为 Kolosov-Muskhelishvili 复势理论下的点力基本解核函数。源点布置为“椭圆孔内部一圈 + 外方形外部一圈”，并保留 \(a_0,a_1,b_0,b_1\) 低阶复势补充项，以表示均匀应力、刚体位移和线性背景场。

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time
import random
import os
from matplotlib.patches import Ellipse, ConnectionPatch

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from IPython.display import display
except ImportError:
    display = print

# ==========================================
# 0. 设备、随机种子与绘图风格
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前使用的计算设备: {device}")


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(42)

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman", "DejaVu Serif"]
plt.rcParams["mathtext.fontset"] = "stix"

# ==========================================
# 1. 物理参数与 KM 常数 (严格对齐 BEM)
# ==========================================
R_major = 0.5        # 椭圆长半轴 a (m)
R_minor = 0.25       # 椭圆短半轴 b (m)
L = 2.0              # 方板边长 (m)
tn = 1.0             # 顶部竖向拉伸面力 (Pa)
E = 1000.0           # 杨氏模量 (Pa)
nu = 0.3             # 泊松比

# 原 BEM 单元采用等效平面应力常数 kappa=(3-nu)/(1+nu)
kappa = (3.0 - nu) / (1.0 + nu)
G = E / (2.0 * (1.0 + nu))

# 源点与配点数量
SOURCE_SCALE = 1.25
SOURCE_OFFSET = 0.5 * (SOURCE_SCALE - 1.0) * L
HOLE_SOURCE_SCALE = 0.70
N_HOLE_SOURCES = 64
N_SOURCE_PER_SIDE = 24
N_HOLE_BND = 360
N_OUT_BND = 160

print("=== KM-PIKFNN 求解有限域椭圆开孔受拉板问题 ===")
print(f"KM constants: kappa={kappa:.6f}, G={G:.6f}")
print(f"Source layout: hole={N_HOLE_SOURCES}, outer={4*N_SOURCE_PER_SIDE}, total={N_HOLE_SOURCES + 4*N_SOURCE_PER_SIDE}")

# 如果已运行 BEM 单元，则保存参考绘图标尺，避免后面 KM 变量覆盖
BEM_UX_VMIN, BEM_UX_VMAX = -0.8, 0.8
BEM_UY_VMIN, BEM_UY_VMAX = 0.10, 3.50
BEM_VM_VMIN = 0.3
if "von_mises" in globals():
    BEM_VM_VMAX = float(np.nanmax(np.asarray(von_mises)))
else:
    BEM_VM_VMAX = 6.3

# ==========================================
# 2. 源点、边界点与源点分布图
# ==========================================
def build_source_points(
    n_sources_per_side=N_SOURCE_PER_SIDE,
    source_offset=SOURCE_OFFSET,
    n_hole_sources=N_HOLE_SOURCES,
    hole_source_scale=HOLE_SOURCE_SCALE,
):
    """生成椭圆孔内部源点环与外方形外部源点环。"""
    theta = np.linspace(0.0, 2.0 * np.pi, int(n_hole_sources), endpoint=False, dtype=np.float32)
    hole_sources = np.column_stack([
        float(hole_source_scale) * R_major * np.cos(theta),
        float(hole_source_scale) * R_minor * np.sin(theta),
    ])

    offset = float(source_offset)
    half_len = 0.5 * L + offset
    lo, hi = -half_len, half_len
    t = np.linspace(lo, hi, int(n_sources_per_side) + 2, dtype=np.float32)[1:-1]

    bottom = np.column_stack([t, np.full_like(t, lo)])
    right = np.column_stack([np.full_like(t, hi), t])
    top = np.column_stack([t[::-1], np.full_like(t, hi)])
    left = np.column_stack([np.full_like(t, lo), t[::-1]])
    outer_sources = np.vstack([bottom, right, top, left])
    return np.vstack([hole_sources, outer_sources]).astype(np.float32)


def split_source_points(source_points, n_hole_sources=N_HOLE_SOURCES):
    n_hole_sources = int(n_hole_sources)
    return source_points[:n_hole_sources], source_points[n_hole_sources:]


def build_branch_angles(source_x, source_y, n_hole_sources=N_HOLE_SOURCES):
    """给每个对数核设置分支方向，避免分支切线穿过物理域。"""
    n_hole_sources = int(n_hole_sources)
    if torch.is_tensor(source_x):
        theta = torch.empty_like(source_x)
        n_total = int(source_x.numel())
        if n_hole_sources > 0:
            # 内部椭圆源点的分支切线指向椭圆孔内部。
            theta[:n_hole_sources] = torch.atan2(
                source_y[:n_hole_sources] / float(R_minor),
                source_x[:n_hole_sources] / float(R_major),
            )
        if n_hole_sources < n_total:
            # 外部源点的分支切线指向远离计算域的一侧。
            theta[n_hole_sources:] = torch.atan2(
                -source_y[n_hole_sources:],
                -source_x[n_hole_sources:],
            )
        return theta

    sx = np.asarray(source_x, dtype=float)
    sy = np.asarray(source_y, dtype=float)
    theta = np.empty_like(sx, dtype=float)
    n_total = int(theta.size)
    if n_hole_sources > 0:
        theta[:n_hole_sources] = np.arctan2(
            sy[:n_hole_sources] / float(R_minor),
            sx[:n_hole_sources] / float(R_major),
        )
    if n_hole_sources < n_total:
        theta[n_hole_sources:] = np.arctan2(
            -sy[n_hole_sources:],
            -sx[n_hole_sources:],
        )
    return theta


def build_boundary_points_numpy(n_hole=N_HOLE_BND, n_out=N_OUT_BND):
    theta = np.linspace(0.0, 2.0 * np.pi, int(n_hole), endpoint=False, dtype=float)
    hole = np.column_stack([R_major * np.cos(theta), R_minor * np.sin(theta)])
    normals = ellipse_unit_normal_numpy(theta)

    t = np.linspace(-L / 2.0, L / 2.0, int(n_out) + 2, dtype=float)[1:-1]
    half = 0.5 * L
    outer = {
        "top": np.column_stack([t, np.full_like(t, half)]),
        "bottom": np.column_stack([t, np.full_like(t, -half)]),
        "right": np.column_stack([np.full_like(t, half), t]),
        "left": np.column_stack([np.full_like(t, -half), t]),
    }
    anchor = np.array([[0.0, -half]], dtype=float)
    return hole, normals, outer, anchor


def ellipse_unit_normal_numpy(theta):
    theta = np.asarray(theta, dtype=float)
    nx = np.cos(theta) / R_major
    ny = np.sin(theta) / R_minor
    norm = np.sqrt(nx * nx + ny * ny)
    return np.column_stack([nx / norm, ny / norm])


def ellipse_unit_normal_torch(theta):
    nx = torch.cos(theta) / float(R_major)
    ny = torch.sin(theta) / float(R_minor)
    norm = torch.sqrt(nx.square() + ny.square())
    return nx / norm, ny / norm


def plot_source_boundary_points(
    n_sources_per_side=N_SOURCE_PER_SIDE,
    n_hole_sources=N_HOLE_SOURCES,
    n_hole=N_HOLE_BND,
    n_out=N_OUT_BND,
    source_offset=SOURCE_OFFSET,
    hole_source_scale=HOLE_SOURCE_SCALE,
):
    """绘制椭圆孔内部源点、外方形源点，并放大显示椭圆孔附近的源点分布。"""
    source_points = build_source_points(
        n_sources_per_side=n_sources_per_side,
        source_offset=source_offset,
        n_hole_sources=n_hole_sources,
        hole_source_scale=hole_source_scale,
    )
    hole_sources, outer_sources = split_source_points(source_points, n_hole_sources)
    hole_bnd, _, outer_bnd, _ = build_boundary_points_numpy(n_hole, n_out)
    outer_boundary = np.vstack([outer_bnd["bottom"], outer_bnd["right"], outer_bnd["top"], outer_bnd["left"]])

    half = 0.5 * L
    source_half = 0.5 * L + float(source_offset)
    physical_square = np.array([
        [-half, -half], [half, -half], [half, half], [-half, half], [-half, -half]
    ])
    source_square = np.array([
        [-source_half, -source_half], [source_half, -source_half],
        [source_half, source_half], [-source_half, source_half],
        [-source_half, -source_half]
    ])

    fig = plt.figure(figsize=(11.4, 5.0), dpi=180)
    gs = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.14], wspace=0.12)
    ax = fig.add_subplot(gs[0, 0])
    ax_zoom = fig.add_subplot(gs[0, 1])

    source_color = "#E60012"
    boundary_color = "#1B2BFF"
    geom_color = "#111827"
    ellipse_color = "#E91E63"

    ax.plot(physical_square[:, 0], physical_square[:, 1], color=geom_color, linewidth=1.7, zorder=1)
    ax.plot(source_square[:, 0], source_square[:, 1], color=geom_color, linewidth=0.9, linestyle="--", alpha=0.68, zorder=1)
    ax.plot(hole_bnd[:, 0], hole_bnd[:, 1], color=geom_color, linewidth=1.2, zorder=2)
    ax.scatter(outer_boundary[:, 0], outer_boundary[:, 1], s=9, color=boundary_color, edgecolors="none", label="Boundary points", zorder=3)
    ax.scatter(hole_bnd[:, 0], hole_bnd[:, 1], s=8, color=boundary_color, edgecolors="none", zorder=3)
    ax.scatter(outer_sources[:, 0], outer_sources[:, 1], s=20, color=source_color, edgecolors="white", linewidths=0.35, label="Source points", zorder=4)
    ax.scatter(hole_sources[:, 0], hole_sources[:, 1], s=24, color=source_color, edgecolors="white", linewidths=0.35, zorder=5)

    zoom_x = 1.55 * R_major
    zoom_y = 1.70 * R_minor
    zoom_box = np.array([
        [-zoom_x, -zoom_y], [zoom_x, -zoom_y],
        [zoom_x, zoom_y], [-zoom_x, zoom_y], [-zoom_x, -zoom_y]
    ])
    ax.plot(zoom_box[:, 0], zoom_box[:, 1], color=geom_color, linestyle="--", linewidth=0.9, zorder=6)

    ax_zoom.plot(hole_bnd[:, 0], hole_bnd[:, 1], color=ellipse_color, linewidth=1.4, zorder=1)
    ax_zoom.scatter(hole_bnd[:, 0], hole_bnd[:, 1], s=11, color=boundary_color, edgecolors="none", label="Boundary points", zorder=3)
    ax_zoom.scatter(hole_sources[:, 0], hole_sources[:, 1], s=36, color=source_color, edgecolors="white", linewidths=0.45, label="Source points", zorder=4)
    ax_zoom.axhline(0.0, color="#777777", linewidth=0.8, alpha=0.65)
    ax_zoom.axvline(0.0, color="#777777", linewidth=0.8, alpha=0.65)

    for xy_global, xy_zoom in [
        ((zoom_x, zoom_y), (-zoom_x, zoom_y)),
        ((zoom_x, -zoom_y), (-zoom_x, -zoom_y)),
    ]:
        con = ConnectionPatch(
            xyA=xy_global, coordsA=ax.transData,
            xyB=xy_zoom, coordsB=ax_zoom.transData,
            color=geom_color, linestyle="--", linewidth=0.8, alpha=0.75
        )
        fig.add_artist(con)

    plot_half = source_half * 1.06
    ax.set_xlim(-plot_half, plot_half)
    ax.set_ylim(-plot_half, plot_half)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel(r"$x$", fontsize=13)
    ax.set_ylabel(r"$y$", fontsize=13)
    ax.set_title("Global source-point distribution", fontsize=13, pad=8)
    ax.tick_params(axis="both", which="major", direction="in", top=True, right=True, length=4.5, width=0.9, labelsize=9)

    ax_zoom.set_xlim(-1.22 * R_major, 1.22 * R_major)
    ax_zoom.set_ylim(-1.45 * R_minor, 1.45 * R_minor)
    ax_zoom.set_aspect("equal", adjustable="box")
    ax_zoom.set_xlabel(r"$x$", fontsize=13)
    ax_zoom.set_ylabel(r"$y$", fontsize=13)
    ax_zoom.set_title("Magnified view near elliptical hole", fontsize=13, pad=8)
    ax_zoom.tick_params(axis="both", which="major", direction="in", top=True, right=True, length=4.5, width=0.9, labelsize=9)

    handles, labels = ax.get_legend_handles_labels()
    fig.legend(
        handles, labels, loc="upper left", bbox_to_anchor=(0.065, 0.985),
        frameon=True, edgecolor="#D1D5DB", facecolor="white", framealpha=0.96,
        fontsize=10, handletextpad=0.45, borderpad=0.42
    )
    fig.subplots_adjust(left=0.07, right=0.985, bottom=0.13, top=0.87)

    print(
        f"Source distribution: {len(hole_sources)} hole interior sources + "
        f"{len(outer_sources)} outer sources = {len(source_points)} total; "
        f"{len(hole_bnd)} hole boundary points + {len(outer_boundary)} outer boundary points."
    )
    plt.show()
    return fig, (ax, ax_zoom)

# ==========================================
# 3. KM 点力基本解网络
# ==========================================
class CVNN(nn.Module):
    """KM 复势点力基本解网络。

    对每个源点 zeta_j 只训练一个复点力 P_j=F_xj+iF_yj。
    Phi_j 与 Psi_j 由同一个 P_j 同时确定，从而保持 Kelvin 点力基本解
    在 KM 复势理论中的物理耦合关系。
    """

    def __init__(
        self,
        n_sources_per_side=N_SOURCE_PER_SIDE,
        source_offset=SOURCE_OFFSET,
        n_hole_sources=N_HOLE_SOURCES,
        hole_source_scale=HOLE_SOURCE_SCALE,
    ):
        super().__init__()
        source_pts = build_source_points(
            n_sources_per_side=n_sources_per_side,
            source_offset=source_offset,
            n_hole_sources=n_hole_sources,
            hole_source_scale=hole_source_scale,
        )
        source_x = torch.tensor(source_pts[:, 0], dtype=torch.float32)
        source_y = torch.tensor(source_pts[:, 1], dtype=torch.float32)
        theta = build_branch_angles(source_x, source_y, n_hole_sources)

        self.n_sources_per_side = int(n_sources_per_side)
        self.source_offset = float(source_offset)
        self.n_hole_sources = int(n_hole_sources)
        self.hole_source_scale = float(hole_source_scale)
        self.n_basis = int(source_x.numel())
        self.n_outer_sources = self.n_basis - self.n_hole_sources
        self.register_buffer("source_x", source_x.view(1, -1))
        self.register_buffer("source_y", source_y.view(1, -1))
        self.register_buffer("rot_cos", torch.cos(theta).view(1, -1))
        self.register_buffer("rot_sin", torch.sin(theta).view(1, -1))
        self.register_buffer("branch_angle", theta.view(1, -1))

        weight_scale = 1.0e-2 / np.sqrt(self.n_basis)
        self.w_force_re = nn.Parameter(weight_scale * torch.randn(self.n_basis, 1))
        self.w_force_im = nn.Parameter(weight_scale * torch.randn(self.n_basis, 1))

        # 低阶解析补充项：Phi=a0+a1*z，Psi=b0+b1*z。
        self.affine_phi = nn.Parameter(torch.zeros(2, 2))
        self.affine_psi = nn.Parameter(torch.zeros(2, 2))

    @staticmethod
    def _add_affine(out_re, out_im, affine, x, y):
        c_re, c_im = affine[0, 0], affine[0, 1]
        a_re, a_im = affine[1, 0], affine[1, 1]
        out_re = out_re + c_re + a_re * x - a_im * y
        out_im = out_im + c_im + a_re * y + a_im * x
        return out_re, out_im

    def forward(self, x, y):
        x_col = x.reshape(-1, 1)
        y_col = y.reshape(-1, 1)
        q_re = x_col - self.source_x
        q_im = y_col - self.source_y

        w_re = self.rot_cos * q_re + self.rot_sin * q_im
        w_im = self.rot_cos * q_im - self.rot_sin * q_re

        scale = np.sqrt(self.n_basis)
        r2 = torch.clamp(q_re.square() + q_im.square(), min=1.0e-30)
        log_q_re = 0.5 * torch.log(r2) / scale
        log_q_im = (torch.atan2(w_im, w_re) + self.branch_angle) / scale

        inv_r2 = 1.0 / r2
        inv_q_re = q_re * inv_r2 / scale
        inv_q_im = -q_im * inv_r2 / scale

        cf = 1.0 / (2.0 * np.pi * (float(kappa) + 1.0))

        # Phi(z) = -P/[2*pi*(kappa+1)] * log(q)
        p_log_re = log_q_re @ self.w_force_re - log_q_im @ self.w_force_im
        p_log_im = log_q_re @ self.w_force_im + log_q_im @ self.w_force_re
        phi_R = -cf * p_log_re
        phi_I = -cf * p_log_im

        # Psi(z) = [kappa*conj(P)*log(q) + P*conj(zeta)/q] / [2*pi*(kappa+1)]
        pbar_log_re = log_q_re @ self.w_force_re + log_q_im @ self.w_force_im
        pbar_log_im = -log_q_re @ self.w_force_im + log_q_im @ self.w_force_re

        zeta_bar_re = self.source_x
        zeta_bar_im = -self.source_y
        zeta_bar_over_q_re = zeta_bar_re * inv_q_re - zeta_bar_im * inv_q_im
        zeta_bar_over_q_im = zeta_bar_re * inv_q_im + zeta_bar_im * inv_q_re

        p_zeta_bar_over_q_re = zeta_bar_over_q_re @ self.w_force_re - zeta_bar_over_q_im @ self.w_force_im
        p_zeta_bar_over_q_im = zeta_bar_over_q_re @ self.w_force_im + zeta_bar_over_q_im @ self.w_force_re

        psi_R = cf * (float(kappa) * pbar_log_re + p_zeta_bar_over_q_re)
        psi_I = cf * (float(kappa) * pbar_log_im + p_zeta_bar_over_q_im)

        phi_R, phi_I = self._add_affine(phi_R, phi_I, self.affine_phi, x_col, y_col)
        psi_R, psi_I = self._add_affine(psi_R, psi_I, self.affine_psi, x_col, y_col)
        return phi_R.reshape_as(x), phi_I.reshape_as(x), psi_R.reshape_as(x), psi_I.reshape_as(x)


def get_phys_fields(case_model, x, y):
    phi_R, phi_I, psi_R, psi_I = case_model(x, y)

    dphi_dx_R = torch.autograd.grad(phi_R, x, grad_outputs=torch.ones_like(phi_R), create_graph=True, retain_graph=True)[0]
    dphi_dx_I = torch.autograd.grad(phi_I, x, grad_outputs=torch.ones_like(phi_I), create_graph=True, retain_graph=True)[0]
    dpsi_dx_R = torch.autograd.grad(psi_R, x, grad_outputs=torch.ones_like(psi_R), create_graph=True, retain_graph=True)[0]
    dpsi_dx_I = torch.autograd.grad(psi_I, x, grad_outputs=torch.ones_like(psi_I), create_graph=True, retain_graph=True)[0]

    d2phi_dx2_R = torch.autograd.grad(dphi_dx_R, x, grad_outputs=torch.ones_like(dphi_dx_R), create_graph=True, retain_graph=True)[0]
    d2phi_dx2_I = torch.autograd.grad(dphi_dx_I, x, grad_outputs=torch.ones_like(dphi_dx_I), create_graph=True, retain_graph=True)[0]

    kappa_f = float(kappa)
    G_f = float(G)

    # 2G(u_x+i*u_y) = kappa*Phi - z*conj(Phi') - conj(Psi)
    z_conj_dphi_R = x * dphi_dx_R + y * dphi_dx_I
    z_conj_dphi_I = y * dphi_dx_R - x * dphi_dx_I
    u_x = (kappa_f * phi_R - z_conj_dphi_R - psi_R) / (2.0 * G_f)
    u_y = (kappa_f * phi_I - z_conj_dphi_I + psi_I) / (2.0 * G_f)

    # sigma_xx + sigma_yy = 4 Re(Phi')
    # sigma_yy - sigma_xx + 2i*sigma_xy = 2(conj(z)*Phi'' + Psi')
    sum_sig = 4.0 * dphi_dx_R
    diff_R = 2.0 * (x * d2phi_dx2_R + y * d2phi_dx2_I + dpsi_dx_R)
    diff_I = 2.0 * (x * d2phi_dx2_I - y * d2phi_dx2_R + dpsi_dx_I)
    sig_xx = 0.5 * (sum_sig - diff_R)
    sig_yy = 0.5 * (sum_sig + diff_R)
    sig_xy = 0.5 * diff_I
    return u_x, u_y, sig_xx, sig_yy, sig_xy

# ==========================================
# 4. 有限方板椭圆孔边界条件
# ==========================================
def _requires_grad_pair(x_val, y_val):
    x = x_val.clone().detach().to(device).requires_grad_(True)
    y = y_val.clone().detach().to(device).requires_grad_(True)
    return x, y


def _boundary_tensors(n_hole=N_HOLE_BND, n_out=N_OUT_BND):
    theta = torch.linspace(0.0, 2.0 * np.pi, int(n_hole) + 1, dtype=torch.float32, device=device)[:-1].view(-1, 1)
    x_hole = R_major * torch.cos(theta)
    y_hole = R_minor * torch.sin(theta)
    nx_hole, ny_hole = ellipse_unit_normal_torch(theta)

    coords = torch.linspace(-L / 2.0, L / 2.0, int(n_out) + 2, dtype=torch.float32, device=device)[1:-1].view(-1, 1)
    half = torch.full_like(coords, L / 2.0)
    outer = {
        "top": (coords.clone(), half.clone()),
        "bottom": (coords.clone(), -half.clone()),
        "right": (half.clone(), coords.clone()),
        "left": (-half.clone(), coords.clone()),
    }
    anchor = (
        torch.tensor([[0.0]], dtype=torch.float32, device=device),
        torch.tensor([[-L / 2.0]], dtype=torch.float32, device=device),
    )
    return x_hole, y_hole, nx_hole, ny_hole, outer, anchor


def get_bnd_loss(case_model, n_hole=N_HOLE_BND, n_out=N_OUT_BND):
    x_hole0, y_hole0, nx_hole, ny_hole, outer, anchor = _boundary_tensors(n_hole, n_out)
    x_hole, y_hole = _requires_grad_pair(x_hole0, y_hole0)
    _, _, sig_xx_h, sig_yy_h, sig_xy_h = get_phys_fields(case_model, x_hole, y_hole)
    tx_hole = sig_xx_h * nx_hole + sig_xy_h * ny_hole
    ty_hole = sig_xy_h * nx_hole + sig_yy_h * ny_hole
    loss_hole = torch.mean(tx_hole.square() + ty_hole.square())

    x_top, y_top = _requires_grad_pair(*outer["top"])
    _, _, _, sig_yy_top, sig_xy_top = get_phys_fields(case_model, x_top, y_top)
    loss_top = torch.mean((sig_yy_top - tn).square() + sig_xy_top.square())

    x_bot, y_bot = _requires_grad_pair(*outer["bottom"])
    _, uy_bot, _, _, sig_xy_bot = get_phys_fields(case_model, x_bot, y_bot)
    loss_bottom = torch.mean((E * uy_bot).square() + sig_xy_bot.square())

    x_left, y_left = _requires_grad_pair(*outer["left"])
    _, _, sig_xx_l, _, sig_xy_l = get_phys_fields(case_model, x_left, y_left)
    x_right, y_right = _requires_grad_pair(*outer["right"])
    _, _, sig_xx_r, _, sig_xy_r = get_phys_fields(case_model, x_right, y_right)
    loss_sides = torch.mean(sig_xx_l.square() + sig_xy_l.square() + sig_xx_r.square() + sig_xy_r.square())

    x_anchor, y_anchor = _requires_grad_pair(*anchor)
    ux_anchor, _, _, _, _ = get_phys_fields(case_model, x_anchor, y_anchor)
    loss_anchor = torch.mean((E * ux_anchor).square())

    return loss_hole + loss_top + loss_bottom + loss_sides + loss_anchor


def get_bnd_loss_components(case_model, n_hole=N_HOLE_BND, n_out=N_OUT_BND):
    x_hole0, y_hole0, nx_hole, ny_hole, outer, anchor = _boundary_tensors(n_hole, n_out)
    x_hole, y_hole = _requires_grad_pair(x_hole0, y_hole0)
    _, _, sig_xx_h, sig_yy_h, sig_xy_h = get_phys_fields(case_model, x_hole, y_hole)
    tx_hole = sig_xx_h * nx_hole + sig_xy_h * ny_hole
    ty_hole = sig_xy_h * nx_hole + sig_yy_h * ny_hole
    loss_hole = torch.mean(tx_hole.square() + ty_hole.square())

    x_top, y_top = _requires_grad_pair(*outer["top"])
    _, _, _, sig_yy_top, sig_xy_top = get_phys_fields(case_model, x_top, y_top)
    loss_top = torch.mean((sig_yy_top - tn).square() + sig_xy_top.square())

    x_bot, y_bot = _requires_grad_pair(*outer["bottom"])
    _, uy_bot, _, _, sig_xy_bot = get_phys_fields(case_model, x_bot, y_bot)
    loss_bottom = torch.mean((E * uy_bot).square() + sig_xy_bot.square())

    x_left, y_left = _requires_grad_pair(*outer["left"])
    _, _, sig_xx_l, _, sig_xy_l = get_phys_fields(case_model, x_left, y_left)
    x_right, y_right = _requires_grad_pair(*outer["right"])
    _, _, sig_xx_r, _, sig_xy_r = get_phys_fields(case_model, x_right, y_right)
    loss_sides = torch.mean(sig_xx_l.square() + sig_xy_l.square() + sig_xx_r.square() + sig_xy_r.square())

    x_anchor, y_anchor = _requires_grad_pair(*anchor)
    ux_anchor, _, _, _, _ = get_phys_fields(case_model, x_anchor, y_anchor)
    loss_anchor = torch.mean((E * ux_anchor).square())

    return {
        "hole": float(loss_hole.detach().cpu().item()),
        "top": float(loss_top.detach().cpu().item()),
        "bottom": float(loss_bottom.detach().cpu().item()),
        "sides": float(loss_sides.detach().cpu().item()),
        "anchor": float(loss_anchor.detach().cpu().item()),
    }

# ==========================================
# 5. 线性最小二乘初始化
# ==========================================
def _kmfs_as_complex(points):
    points = np.asarray(points, dtype=float)
    return points[:, 0] + 1j * points[:, 1]


def _kmfs_coefficient_count(n_basis):
    return 2 * int(n_basis) + 8


def _kmfs_unpack_coeffs(coeffs, n_basis):
    coeffs = np.asarray(coeffs, dtype=float)
    n = int(n_basis)
    idx = 0
    point_force = coeffs[idx:idx+n] + 1j * coeffs[idx+n:idx+2*n]
    idx += 2 * n
    phi0 = coeffs[idx] + 1j * coeffs[idx+1]
    idx += 2
    phi1 = coeffs[idx] + 1j * coeffs[idx+1]
    idx += 2
    psi0 = coeffs[idx] + 1j * coeffs[idx+1]
    idx += 2
    psi1 = coeffs[idx] + 1j * coeffs[idx+1]
    return point_force, phi0, phi1, psi0, psi1


def _kmfs_prepare_cache(points, source_points, n_hole_sources=N_HOLE_SOURCES):
    z = _kmfs_as_complex(points)
    src_c = _kmfs_as_complex(source_points)
    dz = z[:, None] - src_c[None, :]
    branch_angle = build_branch_angles(src_c.real, src_c.imag, n_hole_sources)
    rotated = dz * np.exp(-1j * branch_angle[None, :])

    n_basis = len(src_c)
    scale = np.sqrt(n_basis)
    basis = (
        np.log(np.maximum(np.abs(rotated), 1.0e-300))
        + 1j * (np.angle(rotated) + branch_angle[None, :])
    ) / scale
    d1 = (1.0 / dz) / scale
    d2 = (-1.0 / (dz * dz)) / scale
    return {
        "z": z,
        "basis": basis,
        "d1": d1,
        "d2": d2,
        "zeta_bar": np.conj(src_c),
        "n_basis": n_basis,
    }


def _kmfs_fields_from_coeffs(coeffs, cache):
    z = cache["z"]
    basis = cache["basis"]
    d1 = cache["d1"]
    d2 = cache["d2"]
    zeta_bar = cache["zeta_bar"]
    n_basis = cache["n_basis"]
    point_force, phi0, phi1, psi0, psi1 = _kmfs_unpack_coeffs(coeffs, n_basis)

    cf = 1.0 / (2.0 * np.pi * (float(kappa) + 1.0))
    phi = -cf * (basis @ point_force) + phi0 + phi1 * z
    dphi = -cf * (d1 @ point_force) + phi1
    d2phi = -cf * (d2 @ point_force)

    psi = cf * (
        float(kappa) * (basis @ np.conj(point_force))
        + (d1 * zeta_bar[None, :]) @ point_force
    ) + psi0 + psi1 * z
    dpsi = cf * (
        float(kappa) * (d1 @ np.conj(point_force))
        + (d2 * zeta_bar[None, :]) @ point_force
    ) + psi1

    disp_complex = (kappa * phi - z * np.conj(dphi) - np.conj(psi)) / (2.0 * G)
    ux = disp_complex.real
    uy = disp_complex.imag

    sum_sig = 4.0 * dphi.real
    diff = 2.0 * (np.conj(z) * d2phi + dpsi)
    sig_xx = 0.5 * (sum_sig - diff.real)
    sig_yy = 0.5 * (sum_sig + diff.real)
    sig_xy = 0.5 * diff.imag
    return ux, uy, sig_xx, sig_yy, sig_xy


def _kmfs_boundary_residual_numpy(coeffs, boundary_caches, boundary_points):
    hole_points, hole_normals, outer_points, anchor_point = boundary_points

    _, _, sxx_h, syy_h, sxy_h = _kmfs_fields_from_coeffs(coeffs, boundary_caches["hole"])
    nx = hole_normals[:, 0]
    ny = hole_normals[:, 1]
    tx_hole = sxx_h * nx + sxy_h * ny
    ty_hole = sxy_h * nx + syy_h * ny

    residuals = [tx_hole, ty_hole]

    _, _, _, syy_t, sxy_t = _kmfs_fields_from_coeffs(coeffs, boundary_caches["top"])
    residuals.extend([syy_t - tn, sxy_t])

    _, uy_b, _, _, sxy_b = _kmfs_fields_from_coeffs(coeffs, boundary_caches["bottom"])
    residuals.extend([E * uy_b, sxy_b])

    _, _, sxx_l, _, sxy_l = _kmfs_fields_from_coeffs(coeffs, boundary_caches["left"])
    _, _, sxx_r, _, sxy_r = _kmfs_fields_from_coeffs(coeffs, boundary_caches["right"])
    residuals.extend([sxx_l, sxy_l, sxx_r, sxy_r])

    ux_anchor, _, _, _, _ = _kmfs_fields_from_coeffs(coeffs, boundary_caches["anchor"])
    residuals.append(E * ux_anchor)
    return np.concatenate(residuals)


def solve_kmfsnet_output_layer_lstsq(
    n_sources_per_side=N_SOURCE_PER_SIDE,
    source_offset=SOURCE_OFFSET,
    n_hole_sources=N_HOLE_SOURCES,
    hole_source_scale=HOLE_SOURCE_SCALE,
    n_hole=N_HOLE_BND,
    n_out=N_OUT_BND,
    rcond=None,
    progress=True,
):
    t0 = time.time()
    source_points = build_source_points(
        n_sources_per_side=n_sources_per_side,
        source_offset=source_offset,
        n_hole_sources=n_hole_sources,
        hole_source_scale=hole_source_scale,
    ).astype(float)
    hole_points, hole_normals, outer_points, anchor_point = build_boundary_points_numpy(n_hole, n_out)
    boundary_points = (hole_points, hole_normals, outer_points, anchor_point)
    boundary_caches = {"hole": _kmfs_prepare_cache(hole_points, source_points, n_hole_sources)}
    for name, points in outer_points.items():
        boundary_caches[name] = _kmfs_prepare_cache(points, source_points, n_hole_sources)
    boundary_caches["anchor"] = _kmfs_prepare_cache(anchor_point, source_points, n_hole_sources)

    n_unknown = _kmfs_coefficient_count(len(source_points))
    zero = np.zeros(n_unknown, dtype=float)
    r0 = _kmfs_boundary_residual_numpy(zero, boundary_caches, boundary_points)
    A = np.empty((r0.size, n_unknown), dtype=float)

    if progress:
        print(f"  KM-PIKFNN LS warm start assembly: rows={r0.size}, unknowns={n_unknown}")

    for k in range(n_unknown):
        e = np.zeros(n_unknown, dtype=float)
        e[k] = 1.0
        A[:, k] = _kmfs_boundary_residual_numpy(e, boundary_caches, boundary_points) - r0

    coeffs, residuals, rank, singular_values = np.linalg.lstsq(A, -r0, rcond=rcond)
    final_residual = _kmfs_boundary_residual_numpy(coeffs, boundary_caches, boundary_points)
    elapsed = time.time() - t0
    rms = float(np.sqrt(np.mean(final_residual**2)))
    max_abs = float(np.max(np.abs(final_residual)))
    cond_est = float(singular_values[0] / singular_values[-1]) if singular_values.size and singular_values[-1] > 0 else np.inf
    info = {
        "lstsq_rank": int(rank),
        "lstsq_boundary_rms": rms,
        "lstsq_boundary_max_abs": max_abs,
        "lstsq_condition_est": cond_est,
        "lstsq_solve_time_s": float(elapsed),
    }
    return coeffs, info


def load_kmfsnet_coeffs_into_model(case_model, coeffs):
    n = int(case_model.n_basis)
    point_force, phi0, phi1, psi0, psi1 = _kmfs_unpack_coeffs(coeffs, n)
    target_device = case_model.w_force_re.device
    with torch.no_grad():
        case_model.w_force_re.copy_(torch.as_tensor(point_force.real.reshape(-1, 1), dtype=torch.float32, device=target_device))
        case_model.w_force_im.copy_(torch.as_tensor(point_force.imag.reshape(-1, 1), dtype=torch.float32, device=target_device))
        case_model.affine_phi.copy_(torch.as_tensor([[phi0.real, phi0.imag], [phi1.real, phi1.imag]], dtype=torch.float32, device=target_device))
        case_model.affine_psi.copy_(torch.as_tensor([[psi0.real, psi0.imag], [psi1.real, psi1.imag]], dtype=torch.float32, device=target_device))

# ==========================================
# 6. 训练与预测
# ==========================================
def train_km_pikfnn(
    n_sources_per_side=N_SOURCE_PER_SIDE,
    n_hole_sources=N_HOLE_SOURCES,
    source_offset=SOURCE_OFFSET,
    hole_source_scale=HOLE_SOURCE_SCALE,
    n_hole=N_HOLE_BND,
    n_out=N_OUT_BND,
    use_lstsq_init=True,
    adam_epochs=500,
    lbfgs_max_iter=1200,
    lr=5e-4,
    seed=42,
    verbose=True,
):
    seed_everything(seed)
    case_model = CVNN(
        n_sources_per_side=n_sources_per_side,
        source_offset=source_offset,
        n_hole_sources=n_hole_sources,
        hole_source_scale=hole_source_scale,
    ).to(device)

    t0 = time.time()
    lstsq_info = {}
    if use_lstsq_init:
        coeffs, lstsq_info = solve_kmfsnet_output_layer_lstsq(
            n_sources_per_side=n_sources_per_side,
            source_offset=source_offset,
            n_hole_sources=n_hole_sources,
            hole_source_scale=hole_source_scale,
            n_hole=n_hole,
            n_out=n_out,
            rcond=None,
            progress=verbose,
        )
        load_kmfsnet_coeffs_into_model(case_model, coeffs)
        if verbose:
            print(
                "  KM-PIKFNN LS init | "
                f"rank={lstsq_info['lstsq_rank']} | "
                f"rms={lstsq_info['lstsq_boundary_rms']:.4e} | "
                f"max={lstsq_info['lstsq_boundary_max_abs']:.4e} | "
                f"cond~={lstsq_info['lstsq_condition_est']:.4e}"
            )

    if int(adam_epochs) > 0:
        optimizer = torch.optim.Adam(case_model.parameters(), lr=lr)
        if verbose:
            print(f"--- Adam fine-tune from LS init ({adam_epochs} epochs) ---")
        for epoch in range(int(adam_epochs)):
            optimizer.zero_grad()
            loss = get_bnd_loss(case_model, n_hole, n_out)
            loss.backward()
            optimizer.step()
            if verbose and (epoch % max(1, adam_epochs // 5) == 0 or epoch == adam_epochs - 1):
                print(f"  Adam {epoch:5d}/{adam_epochs} | loss={loss.item():.4e}")

    if int(lbfgs_max_iter) > 0:
        if verbose:
            print(f"--- L-BFGS fine-tune from current state (max_iter={lbfgs_max_iter}) ---")
        lbfgs_optimizer = torch.optim.LBFGS(
            case_model.parameters(),
            lr=1.0,
            max_iter=int(lbfgs_max_iter),
            max_eval=int(lbfgs_max_iter * 1.5) + 1,
            history_size=100,
            tolerance_grad=1e-9,
            tolerance_change=1e-11,
            line_search_fn="strong_wolfe",
        )
        iter_count = [0]

        def closure():
            lbfgs_optimizer.zero_grad()
            loss = get_bnd_loss(case_model, n_hole, n_out)
            loss.backward()
            iter_count[0] += 1
            if verbose and iter_count[0] % max(1, lbfgs_max_iter // 5) == 0:
                print(f"  L-BFGS {iter_count[0]:5d} | loss={loss.item():.4e}")
            return loss

        lbfgs_optimizer.step(closure)

    final_loss = float(get_bnd_loss(case_model, n_hole, n_out).detach().cpu().item())
    components = get_bnd_loss_components(case_model, n_hole, n_out)
    elapsed = time.time() - t0
    return case_model, final_loss, elapsed, lstsq_info, components


def predict_km_fields_on_points(case_model, pts, batch_size=10000):
    case_model.eval()
    ux_chunks, uy_chunks = [], []
    sxx_chunks, syy_chunks, sxy_chunks = [], [], []
    n_pts = pts.shape[0]
    for start in range(0, n_pts, batch_size):
        end = min(start + batch_size, n_pts)
        batch = pts[start:end]
        x = torch.tensor(batch[:, 0:1], dtype=torch.float32, device=device, requires_grad=True)
        y = torch.tensor(batch[:, 1:2], dtype=torch.float32, device=device, requires_grad=True)
        ux, uy, sig_xx, sig_yy, sig_xy = get_phys_fields(case_model, x, y)

        ux_chunks.append(ux.detach().cpu().numpy().reshape(-1))
        uy_chunks.append(uy.detach().cpu().numpy().reshape(-1))
        sxx_chunks.append(sig_xx.detach().cpu().numpy().reshape(-1))
        syy_chunks.append(sig_yy.detach().cpu().numpy().reshape(-1))
        sxy_chunks.append(sig_xy.detach().cpu().numpy().reshape(-1))
        del x, y, ux, uy, sig_xx, sig_yy, sig_xy
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    ux = np.concatenate(ux_chunks)
    uy = np.concatenate(uy_chunks)
    sig_xx = np.concatenate(sxx_chunks)
    sig_yy = np.concatenate(syy_chunks)
    sig_xy = np.concatenate(sxy_chunks)
    sig_zz = 0.0
    von = np.sqrt(
        0.5 * ((sig_xx - sig_yy)**2 + (sig_yy - sig_zz)**2 + (sig_zz - sig_xx)**2)
        + 3.0 * sig_xy**2
    )
    return {
        "ux": ux,
        "uy": uy,
        "sig_xx": sig_xx,
        "sig_yy": sig_yy,
        "sig_xy": sig_xy,
        "von_mises": von,
    }


def make_field_map(values, mask, grid_size):
    out = np.full(grid_size * grid_size, np.nan)
    out[np.asarray(mask).reshape(-1)] = np.asarray(values).reshape(-1)
    return out.reshape(grid_size, grid_size)


def draw_field_map(field_map, title, cbar_label, vmin=None, vmax=None, cmap="jet"):
    fig, ax = plt.subplots(figsize=(6.2, 5.2), dpi=150)
    extent = [-L_plot, L_plot, -L_plot, L_plot]
    im = ax.imshow(field_map, origin="lower", extent=extent, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_title(title)
    ax.set_aspect("equal", adjustable="box")
    ax.add_patch(Ellipse((0, 0), 2 * R_major, 2 * R_minor, color="white", fill=True, zorder=3))
    ax.add_patch(Ellipse((0, 0), 2 * R_major, 2 * R_minor, color="black", fill=False, linewidth=1.2, zorder=4))
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(cbar_label)
    plt.tight_layout()
    plt.show()
    return fig, ax

# ==========================================
# 7. 执行训练、后处理与绘图
# ==========================================
if __name__ == "__main__":
    plot_source_boundary_points(
        n_sources_per_side=N_SOURCE_PER_SIDE,
        n_hole_sources=N_HOLE_SOURCES,
        n_hole=N_HOLE_BND,
        n_out=N_OUT_BND,
        source_offset=SOURCE_OFFSET,
        hole_source_scale=HOLE_SOURCE_SCALE,
    )

    model, final_loss, train_time_s, lstsq_info, final_components = train_km_pikfnn(
        n_sources_per_side=N_SOURCE_PER_SIDE,
        n_hole_sources=N_HOLE_SOURCES,
        source_offset=SOURCE_OFFSET,
        hole_source_scale=HOLE_SOURCE_SCALE,
        n_hole=N_HOLE_BND,
        n_out=N_OUT_BND,
        use_lstsq_init=True,
        adam_epochs=500,
        lbfgs_max_iter=1200,
        lr=5e-4,
        seed=42,
        verbose=True,
    )

    print(f"L-BFGS 完成 | 最终 Loss: {final_loss:.4e}")
    print(f"训练总耗时: {train_time_s:.2f} s")
    print("Final boundary loss components:", final_components)

    L_plot = L / 2.0
    grid_size = 200
    coords_1d = np.linspace(-L_plot, L_plot, grid_size)
    XX, YY = np.meshgrid(coords_1d, coords_1d)
    ellipse_val = (XX / R_major)**2 + (YY / R_minor)**2
    valid_mask = ellipse_val >= (BEM_PLOT_MASK_LEVEL if 'BEM_PLOT_MASK_LEVEL' in globals() else 1.03)
    eval_pts = np.c_[XX[valid_mask].ravel(), YY[valid_mask].ravel()]

    km_fields = predict_km_fields_on_points(model, eval_pts, batch_size=10000)
    KM_U_X_map = make_field_map(km_fields["ux"] * 1e3, valid_mask, grid_size)
    KM_U_Y_map = make_field_map(km_fields["uy"] * 1e3, valid_mask, grid_size)
    KM_VM_map = make_field_map(km_fields["von_mises"], valid_mask, grid_size)

    draw_field_map(KM_U_X_map, r"KM-PIKFNN: $U_x$ prediction", "mm", vmin=BEM_UX_VMIN, vmax=BEM_UX_VMAX)
    draw_field_map(KM_U_Y_map, r"KM-PIKFNN: $U_y$ prediction", "mm", vmin=BEM_UY_VMIN, vmax=BEM_UY_VMAX)
    draw_field_map(KM_VM_map, "KM-PIKFNN: von Mises stress", "Pa", vmin=BEM_VM_VMIN, vmax=BEM_VM_VMAX)

    km_summary = {
        "method": "KM-PIKFNN",
        "hole_boundary_points": int(N_HOLE_BND),
        "outer_boundary_points": int(4 * N_OUT_BND),
        "total_boundary_points": int(N_HOLE_BND + 4 * N_OUT_BND),
        "hole_source_points": int(N_HOLE_SOURCES),
        "outer_source_points": int(4 * N_SOURCE_PER_SIDE),
        "total_source_points": int(N_HOLE_SOURCES + 4 * N_SOURCE_PER_SIDE),
        "trainable_parameters": int(2 * (N_HOLE_SOURCES + 4 * N_SOURCE_PER_SIDE) + 8),
        "final_boundary_loss": float(final_loss),
        "train_time_s": float(train_time_s),
    }
    print("KM-PIKFNN summary:", km_summary)

In [ ]:
# ==========================================
# 8. KM-PIKFNN 相对于 BEM 参考解的位移与 von Mises 误差
# ==========================================
print("\n=== 计算 KM-PIKFNN 与 BEM 参考解之间的误差 ===")

required_names = ["model", "internal_pts", "u_int", "von_mises", "valid_mask", "grid_size", "L_plot"]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise NameError(f"请先依次运行 BEM 单元和 KM-PIKFNN 训练单元，当前缺少: {missing}")

km_ref_fields = predict_km_fields_on_points(model, internal_pts, batch_size=10000)
u_pred_real = km_ref_fields["ux"]
v_pred_real = km_ref_fields["uy"]
vm_pred_real = km_ref_fields["von_mises"]

u_bem_ref = np.asarray(u_int[:, 0]).reshape(-1)
v_bem_ref = np.asarray(u_int[:, 1]).reshape(-1)
vm_bem_ref = np.asarray(von_mises).reshape(-1)

n_eval = internal_pts.shape[0]
for name, arr in {
    "u_pred_real": u_pred_real,
    "v_pred_real": v_pred_real,
    "vm_pred_real": vm_pred_real,
    "u_bem_ref": u_bem_ref,
    "v_bem_ref": v_bem_ref,
    "vm_bem_ref": vm_bem_ref,
}.items():
    if arr.size != n_eval:
        raise ValueError(f"{name} 长度为 {arr.size}，但 internal_pts 长度为 {n_eval}，请确认 BEM 与 KM-PIKFNN 使用同一批评估点。")

norm_factor_ux = max(np.max(np.abs(u_bem_ref)), 1e-15)
norm_factor_uy = max(np.max(np.abs(v_bem_ref)), 1e-15)
norm_factor_vm = max(np.max(np.abs(vm_bem_ref)), 1e-15)

err_ux_norm = np.abs(u_pred_real - u_bem_ref) / norm_factor_ux
err_uy_norm = np.abs(v_pred_real - v_bem_ref) / norm_factor_uy
err_vm_norm = np.abs(vm_pred_real - vm_bem_ref) / norm_factor_vm

L2_ux = np.linalg.norm(u_pred_real - u_bem_ref, 2) / max(np.linalg.norm(u_bem_ref, 2), 1e-15)
L2_uy = np.linalg.norm(v_pred_real - v_bem_ref, 2) / max(np.linalg.norm(v_bem_ref, 2), 1e-15)
L2_vm = np.linalg.norm(vm_pred_real - vm_bem_ref, 2) / max(np.linalg.norm(vm_bem_ref, 2), 1e-15)

print("-" * 62)
print(f"Relative L2 error Ux        : {L2_ux:.3e}")
print(f"Relative L2 error Uy        : {L2_uy:.3e}")
print(f"Relative L2 error von Mises : {L2_vm:.3e}")
print(f"Max Normalized Error Ux        : {err_ux_norm.max():.3e}")
print(f"Max Normalized Error Uy        : {err_uy_norm.max():.3e}")
print(f"Max Normalized Error von Mises : {err_vm_norm.max():.3e}")
print("-" * 62)

valid_mask_1d = np.asarray(valid_mask).reshape(-1)
extent = [-L_plot, L_plot, -L_plot, L_plot]


def _make_error_map(error_values):
    error_map = np.full(grid_size * grid_size, np.nan)
    error_map[valid_mask_1d] = np.asarray(error_values).reshape(-1)
    return error_map.reshape(grid_size, grid_size)


Err_X_map = _make_error_map(err_ux_norm)
Err_Y_map = _make_error_map(err_uy_norm)
Err_VM_map = _make_error_map(err_vm_norm)


def plot_normalized_error(error_map, title):
    fig, ax = plt.subplots(figsize=(6.2, 5.2), dpi=150)
    im = ax.imshow(error_map, origin="lower", extent=extent, cmap="Reds", vmin=0)
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_title(title)
    ax.set_aspect("equal", adjustable="box")
    ax.add_patch(Ellipse((0, 0), 2 * R_major, 2 * R_minor, color="white", fill=True, zorder=3))
    ax.add_patch(Ellipse((0, 0), 2 * R_major, 2 * R_minor, color="black", fill=False, linewidth=1.2, zorder=4))
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Normalized absolute error")
    plt.tight_layout()
    plt.show()
    return fig, ax


plot_normalized_error(Err_X_map, r"Normalized Absolute Error of $U_x$")
plot_normalized_error(Err_Y_map, r"Normalized Absolute Error of $U_y$")
plot_normalized_error(Err_VM_map, "Normalized Absolute Error of von Mises Stress")

km_error_summary = {
    "method": "KM-PIKFNN",
    "internal_pde_points": 0,
    "hole_boundary_points": int(N_HOLE_BND),
    "outer_boundary_points": int(4 * N_OUT_BND),
    "total_boundary_points": int(N_HOLE_BND + 4 * N_OUT_BND),
    "source_points": int(N_HOLE_SOURCES + 4 * N_SOURCE_PER_SIDE),
    "trainable_parameters": int(2 * (N_HOLE_SOURCES + 4 * N_SOURCE_PER_SIDE) + 8),
    "L2_ux": float(L2_ux),
    "L2_uy": float(L2_uy),
    "L2_von_mises": float(L2_vm),
    "train_time_s": float(train_time_s) if "train_time_s" in globals() else np.nan,
}

if pd is not None:
    display(pd.DataFrame([km_error_summary]))
else:
    print(km_error_summary)